# 🗑️ Klasifikasi Sampah — Notebook Pelatihan Model (Tahap 1)

**UAS Pembelajaran Mesin — Primakara University**

Dijalankan di **Google Colab** (aktifkan GPU: *Runtime → Change runtime type → T4 GPU*).

Alur lengkap:
1. Persiapan data (unduh → bersihkan → split → augmentasi)
2. **Optimasi #1 — Hyperparameter Tuning** (Keras Tuner mencari `dropout`, `learning_rate`, unit dense terbaik)
3. Latih kepala model dengan hyperparameter terbaik (base dibekukan)
4. Evaluasi baseline (accuracy, F1, confusion matrix)
5. **Optimasi #2 — Fine-tuning** (buka 30 layer atas MobileNetV2) → evaluasi ulang
6. Simpan model (`model_sampah.h5` + `class_names.json`)

> Konsistensi: normalisasi gambar di sini (`x/127.5 - 1`) **sama persis** dengan
> `src/preprocessing.py` yang dipakai aplikasi.

## Langkah 0 — Import & cek lingkungan

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "TIDAK ADA (latihan akan lambat)")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = (224, 224)   # ukuran input MobileNetV2
BATCH = 32

## Langkah 0b — Login Kaggle (untuk unduh dataset)

Dataset diunduh dari Kaggle, jadi perlu autentikasi **sekali**.

**Dapatkan token:** kaggle.com → login → foto profil → **Settings → API →
Create New API Token** → terunduh `kaggle.json` (berisi `username` & `key`).

Jalankan sel di bawah, lalu **tempel `username` dan `key`** saat diminta.
Jika kotak input tidak muncul, pakai **Secrets** Colab (ikon 🔑 di kiri):
tambahkan `KAGGLE_USERNAME` & `KAGGLE_KEY`.

In [ ]:
import kagglehub
kagglehub.login()   # tempel username & key dari kaggle.json saat diminta

## Langkah 1 — Unduh & temukan dataset

Dataset **Garbage Classification** (TrashNet, 6 kelas) dari Kaggle.

In [ ]:
# Unduh via kagglehub (otomatis di Colab). Jika gagal, lihat data/README.md untuk cara manual.
import kagglehub
download_path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
print("Lokasi unduhan:", download_path)

In [ ]:
# Cari folder yang BERISI langsung 6 subfolder kelas (struktur dataset bisa bersarang).
EXPECTED = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}

def find_data_dir(root):
    root = pathlib.Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for d in candidates:
        subdirs = {s.name.lower() for s in d.iterdir() if s.is_dir()}
        if len(EXPECTED & subdirs) >= 5:
            return d
    raise FileNotFoundError("Folder 6 kelas tidak ditemukan di " + str(root))

DATA_DIR = find_data_dir(download_path)
print("Folder dataset:", DATA_DIR)
print("Kelas:", sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]))

## Langkah 1b — Bersihkan gambar rusak (data cleaning)

In [ ]:
from PIL import Image
removed = 0
for img_path in DATA_DIR.rglob("*"):
    if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        try:
            Image.open(img_path).verify()
        except Exception:
            img_path.unlink(); removed += 1
print("Gambar rusak dibuang:", removed)

## Langkah 1c — Muat dataset & split 70% / 15% / 15% (train / val / test)

In [ ]:
# image_dataset_from_directory hanya bisa split 2 arah, jadi:
#   - train = 70% ; sisanya 30% dipecah lagi jadi val (15%) & test (15%).
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.30, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int",
)
rest_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.30, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int",
)

class_names = train_ds_raw.class_names      # urutan alfabetis = urutan indeks model
num_classes = len(class_names)
print("Kelas (urut indeks):", class_names)

rest_batches = tf.data.experimental.cardinality(rest_ds_raw).numpy()
val_ds_raw  = rest_ds_raw.take(rest_batches // 2)
test_ds_raw = rest_ds_raw.skip(rest_batches // 2)
print("Batch -> train:", tf.data.experimental.cardinality(train_ds_raw).numpy(),
      "| val:", tf.data.experimental.cardinality(val_ds_raw).numpy(),
      "| test:", tf.data.experimental.cardinality(test_ds_raw).numpy())

## Langkah 1d — EDA: distribusi kelas & contoh gambar

In [ ]:
counts = {c: len(list((DATA_DIR / c).glob("*"))) for c in class_names}
plt.figure(figsize=(7, 4))
plt.bar(counts.keys(), counts.values(), color="seagreen")
plt.title("Distribusi jumlah gambar per kelas"); plt.ylabel("jumlah"); plt.xticks(rotation=20)
for i, v in enumerate(counts.values()):
    plt.text(i, v + 5, str(v), ha="center")
plt.tight_layout(); plt.show()
print(counts)
print("Catatan: kelas 'trash' paling sedikit -> ada sedikit class imbalance.")

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds_raw.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

## Langkah 1e — Pipeline: augmentasi (train) + normalisasi (semua)

- **Augmentasi** (flip/rotasi/zoom) hanya untuk **train** → menambah variasi.
- **Normalisasi** `x/127.5 - 1` untuk SEMUA split → **identik** dengan `src/preprocessing.py`.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
], name="augmentation")

def normalize(x, y):
    return x / 127.5 - 1.0, y          # [0,255] -> [-1,1]

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (train_ds_raw
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .map(normalize, num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds  = val_ds_raw.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds_raw.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## Helper — fungsi evaluasi & plot (dipakai berkali-kali)

In [ ]:
def plot_history(h, title=""):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h.history["accuracy"], label="train"); plt.plot(h.history["val_accuracy"], label="val")
    plt.title("Accuracy " + title); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(h.history["loss"], label="train"); plt.plot(h.history["val_loss"], label="val")
    plt.title("Loss " + title); plt.legend()
    plt.tight_layout(); plt.show()

def evaluate(model, ds, title):
    y_true, y_prob = [], []
    for xb, yb in ds:
        y_true.append(yb.numpy()); y_prob.append(model.predict(xb, verbose=0))
    y_true = np.concatenate(y_true); y_prob = np.concatenate(y_prob)
    y_pred = y_prob.argmax(1)
    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"=== {title} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1 macro : {f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix — " + title); plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.ylabel("Aktual"); plt.xlabel("Prediksi"); plt.tight_layout(); plt.show()
    return acc, f1

## Langkah 2 — OPTIMASI #1: Hyperparameter Tuning (Keras Tuner)

Daripada menebak `dropout`, `learning_rate`, dan ukuran lapisan dense, kita cari
otomatis kombinasi terbaik dengan **Keras Tuner (Hyperband)**. Hyperband melatih
banyak kombinasi secara singkat, lalu hanya kombinasi paling menjanjikan yang
dilanjutkan ke lebih banyak epoch — efisien.

> `keras-tuner` hanya dipakai untuk PELATIHAN (tidak masuk requirements.txt app).

In [ ]:
!pip install -q keras-tuner
import keras_tuner as kt
print("keras-tuner:", kt.__version__)

In [ ]:
def build_model(hp):
    # MobileNetV2 (ImageNet) sebagai ekstraktor fitur; base DIBEKUKAN saat tuning.
    base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    # --- hyperparameter yang dicari ---
    x = layers.Dense(hp.Choice("dense_units", [64, 128, 256]), activation="relu")(x)
    x = layers.Dropout(hp.Float("dropout", 0.2, 0.5, step=0.1))(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(base.input, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("lr", [1e-3, 5e-4, 1e-4])),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=10,
    factor=3,
    directory="kt_dir",
    project_name="sampah",
    overwrite=True,
)

stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)
tuner.search(train_ds, validation_data=val_ds, epochs=10, callbacks=[stop_early])

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n>>> HYPERPARAMETER TERBAIK <<<")
print("  dense_units :", best_hp.get("dense_units"))
print("  dropout     :", best_hp.get("dropout"))
print("  learning_rate:", best_hp.get("lr"))

## Langkah 3 — Latih kepala model dengan hyperparameter terbaik (base beku)

In [ ]:
model = tuner.hypermodel.build(best_hp)      # model dengan konfigurasi terbaik
early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[early])
plot_history(history, "(head, hyperparameter terbaik)")

## Langkah 4 — Evaluasi BASELINE (setelah tuning, sebelum fine-tuning)

In [ ]:
acc_base, f1_base = evaluate(model, test_ds, "BASELINE (tuned, base beku)")

## Langkah 5 — OPTIMASI #2: Fine-tuning

Buka **30 layer teratas** MobileNetV2 lalu latih ulang dengan **learning rate
sangat kecil** (1e-5). Layer **BatchNormalization** tetap dibekukan (praktik
standar fine-tuning).

In [ ]:
model.trainable = True
# Buka 30 layer teratas MobileNetV2 untuk fine-tuning. Kita set trainable
# SECARA EKSPLISIT per-layer (lebih andal daripada mengandalkan
# model.trainable=True yang tidak menembus sub-model yang sempat dibekukan).
gap_idx = next(i for i, l in enumerate(model.layers) if isinstance(l, layers.GlobalAveragePooling2D))
feature_layers = model.layers[:gap_idx]      # seluruh layer MobileNetV2

for layer in feature_layers[:-30]:           # bagian bawah: tetap beku
    layer.trainable = False
for layer in feature_layers[-30:]:           # 30 teratas: dilatih, KECUALI BatchNorm
    layer.trainable = not isinstance(layer, layers.BatchNormalization)
for layer in model.layers[gap_idx:]:         # kepala (head): selalu dilatih
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),   # LR kecil!
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

trainable_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
print("Fine-tuning -> layer trainable:", sum(l.trainable for l in model.layers),
      "| parameter trainable:", f"{trainable_params:,}")

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early])
plot_history(history_ft, "(fine-tuning)")

## Langkah 6 — Evaluasi SETELAH fine-tuning & bandingkan

In [ ]:
acc_ft, f1_ft = evaluate(model, test_ds, "SETELAH FINE-TUNING")

print("PERBANDINGAN (bukti optimasi berhasil):")
print(f"  Accuracy : {acc_base:.4f}  ->  {acc_ft:.4f}  (Δ {acc_ft-acc_base:+.4f})")
print(f"  F1 macro : {f1_base:.4f}  ->  {f1_ft:.4f}  (Δ {f1_ft-f1_base:+.4f})")

## Langkah 7 — Simpan model

- `model_sampah.h5` → format yang diminta soal untuk deep learning.
- `class_names.json` → urutan label agar app tahu indeks → nama kelas.
- `model_sampah.tflite` → versi ringan (fallback jika RAM Streamlit Cloud mepet).

In [ ]:
model.save("model_sampah.h5")
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("Tersimpan: model_sampah.h5, class_names.json")

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    open("model_sampah.tflite", "wb").write(converter.convert())
    print("Tersimpan: model_sampah.tflite")
except Exception as e:
    print("Lewati TFLite:", e)

In [ ]:
# Unduh ke komputer (lalu taruh di folder model/ pada repo).
try:
    from google.colab import files
    files.download("model_sampah.h5")
    files.download("class_names.json")
except Exception as e:
    print("Bukan di Colab / unduh manual:", e)

---
### ✅ Selesai Tahap 1
Dua upaya optimasi telah dilakukan: **(1) hyperparameter tuning (Keras Tuner)**
dan **(2) fine-tuning**. Letakkan `model_sampah.h5` & `class_names.json` ke folder
**`model/`** di repository, lalu jalankan `streamlit run app.py`.